In [ ]:
# # ML Service Authentication Guide
#
# This notebook demonstrates how to securely use authentication credentials for Hugging Face and Argilla.
#
# ## 1. Prerequisites & Installation
#
# Before running this notebook, ensure you have initialized your project and installed the required non-standard libraries using `uv`.
#
# In vscode open your terminal and run:
#
# ```bash
# uv add python-dotenv huggingface_hub argilla
# ```
#
# ## 2. Retrieving Credentials
#
# ### Hugging Face Hub Token
# 1.  Navigate to your Hugging Face [Settings](https://huggingface.co/settings/tokens).
# 2.  Select **Access Tokens** from the left menu.
# 3.  Click **Create new token** (select 'Write' permissions if you plan to upload data).
# 4.  Copy the generated token string starting with `hf_`.
#
# ### Argilla API Credentials
# 1.  Open your Argilla instance in the browser.
# 2.  Click on your profile picture or initials in the top right corner.
# 3.  Select **My Settings**.
# 4.  Copy the **API Key**.
#
# ---
#
# ## 3. Storage Setup (.env)
#
# Create a file named `.env` in your project root and populate it as follows (replace the values with your actual credentials and remove the comment symbols `#` from the lines):
#

# HUGGING_FACE_HUB_TOKEN=hf_YourActualTokenHere
# ARGILLA_API_URL=https://nlp.ilsp.gr/argilla
# ARGILLA_API_KEY=your-api-key-here
# ARGILLA_USERNAME=your_username



import logging
from pathlib import Path
from dotenv import dotenv_values, find_dotenv
from huggingface_hub import HfApi, login
import argilla as rg
from typing import Dict

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def load_credentials_from_env() -> Dict[str, str]:
    """
    Locates the .env file using find_dotenv and loads environment variables.
    The function assumes the filename is '.env'.
    """
    # Use find_dotenv to search for the file. 
    # By default, find_dotenv searches for '.env'
    env_path = find_dotenv(usecwd=True, raise_error_if_not_found=False)

    if not env_path:
        # Logging the error message for the missing file at the current working directory, 
        # maintaining consistency with the original notebook's logging style.
        logging.error(f"Configuration file missing at: {Path.cwd() / '.env'}") 
        return {}

    # Convert the found path to a Path object for consistency 
    env_path_obj = Path(env_path)
    
    # Check if the file actually exists
    if not env_path_obj.exists():
        logging.error(f"Configuration file missing at: {env_path_obj}")
        return {}
        
    return dotenv_values(env_path_obj)

def connect_hugging_face(env_values: dict):
    """
    Authenticates with Hugging Face Hub and verifies identity.
    """
    hf_token = env_values.get("HUGGING_FACE_HUB_TOKEN")
    
    if not hf_token:
        logging.warning("HF Token not found in configuration.")
        return

    try:
        # Login to local machine
        login(token=hf_token, add_to_git_credential=False)
        
        # Verify identity via API
        hf_api = HfApi(token=hf_token)
        user_info = hf_api.whoami()
        logging.info(f"HF Authentication Successful: Logged in as '{user_info.get('name')}'")
    except Exception as e:
        logging.error(f"HF Authentication failed: {e}")

def connect_argilla(env_values: dict):
    """
    Initializes Argilla client and verifies connection.
    """
    argilla_url = env_values.get("ARGILLA_API_URL")
    argilla_key = env_values.get("ARGILLA_API_KEY")

    if not argilla_url or not argilla_key:
        logging.warning("Argilla credentials incomplete.")
        return

    try:
        # Initialize Client
        client = rg.Argilla(api_url=argilla_url, api_key=argilla_key)        
    except Exception as e:
        logging.error(f"Argilla connection failed: {e}")

def main():
    # Load configuration
    logging.info("Loading credentials from .env file.")
    env_values = load_credentials_from_env()
    
    if env_values:
        logging.info("Checking Hugging Face connection.")
        connect_hugging_face(env_values)
        
        logging.info("Checking Argilla connection.")
        connect_argilla(env_values)
        
    else:
        logging.warning("Skipping connection attempts due to missing config.")

if __name__ == "__main__":
    main()